In [2]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model # langchain 은 langgraph의 자매 프로젝트 같은 것, 단지 ai 모델이랑 쉽게 대화할 수 잇게 해주는
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini") 

conn = sqlite3.connect(
    "memory.db",
    check_same_thread=False,
)

config = {
    "configurable" : {
        "thread_id" : "11"  # 새로운 걸 할거면 계속 바꿔줘야함 .
    }
}


In [3]:
class State(MessagesState):
    pass


graph_builder = StateGraph(State)

In [4]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [5]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [8]:
result = graph.invoke(
    {
        "messages" : [
            {"role" : "user", "content" : "I ive in Korea now. And the city I live in is Seoul"}
        ]
    },
    config = config 
)


In [9]:
for message in result["messages"]:
    message.pretty_print() # messages가 langchain 안에서 가지고 있는 메서드인데 메시지를 멋지게 보여준다 .

================================ Human Message =================================

Hello my name is Hakyung
================================== Ai Message ==================================

Hello, Hakyung! How can I assist you today?
================================ Human Message =================================

I ive in Korea now. And the city I live in is Seoul
================================== Ai Message ==================================

That's great to hear! Seoul is a vibrant city with a rich history and culture. Do you have any specific places you love in Seoul, or anything you’d like to know about the city?


In [27]:
# state 기록 가져와서 수정해보기 
state_history =  graph.get_state_history(config) # config 에는 thread_id가 있기에 넘긴다. 

In [ ]:
for state_snapshot in list(state_history):
    print(state_snapshot.next) # 현재 ? 
    print(state_snapshot.values["messages"])  # 수정하고자 하는 메시지
    print("==========\n")  # 아래부터 위로 보는거임 .,(-1, -2, -3 .. )

()
[HumanMessage(content='Hello my name is Hakyung', additional_kwargs={}, response_metadata={}, id='77719399-b205-4cd5-8c47-8e63986a2168'), AIMessage(content='Hello, Hakyung! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DA6WpYwWJQQ6KNxPNvOFIpMfVdTtz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c69c0-7349-74b0-a5d6-aaad87c117af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'o

In [ ]:
state_history = graph.get_state_history(config)

# 수정 순서1 ) 과거의 수정 시점으로 돌아감(to_fork)
to_fork = list(state_history)[-5] # 수정하고 싶은 메시지를 저장 (아래에서 부터 세서 -1,-2.. )
to_fork.values["messages"] # 잘 저장됐나 메시지 재확인(여기에는 메시지 아이디 있음 이걸 아래에 넣는것) 
#to_fork.config # 해당 시점 체크포인트 아이디 나온다. (처음에 선언한 config는 thread_id만 가지고있음)

[HumanMessage(content='Hello my name is Hakyung', additional_kwargs={}, response_metadata={}, id='77719399-b205-4cd5-8c47-8e63986a2168'),
 AIMessage(content='Hello, Hakyung! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DA6WpYwWJQQ6KNxPNvOFIpMfVdTtz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c69c0-7349-74b0-a5d6-aaad87c117af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

In [35]:
from langchain_core.messages import HumanMessage

# 수정순서2) 체크포인트 id 설정에서 수정 
graph.update_state(
    to_fork.config,
    {
        "messages" : [
            HumanMessage(
                content = "I ive in Korea now. And the city I live in gyeonggi-do",
                id ="8a177a93-8e97-4011-a231-dca636e667e0",
            )
        ]
    },
)

{'configurable': {'thread_id': '11',
  'checkpoint_ns': '',
  'checkpoint_id': '1f10bbca-0068-6c4f-8004-06feaf6dde2b'}}

In [ ]:
forked_state = graph.get_state_history(  #위에 결과 값 그대로 사용 
   {
        "configurable": {
            "thread_id": "11",
            "checkpoint_ns": "",
            "checkpoint_id": "1f10bbca-0068-6c4f-8004-06feaf6dde2b"
        }
    }
)
list(forked_state) # ㅈ수정된 메시지로 되어있는 것을 볼 수 잇다. 

[StateSnapshot(values={'messages': [HumanMessage(content='Hello my name is Hakyung', additional_kwargs={}, response_metadata={}, id='77719399-b205-4cd5-8c47-8e63986a2168'), AIMessage(content='Hello, Hakyung! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DA6WpYwWJQQ6KNxPNvOFIpMfVdTtz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c69c0-7349-74b0-a5d6-aaad87c117af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {

In [39]:
# 위에 checkpoint id를 사용해서 그래프를 호출 ! !

result = graph.invoke(
    None, 
    {
        "configurable": {
            "thread_id": "11",
            "checkpoint_ns": "",
            "checkpoint_id": "1f10bbca-0068-6c4f-8004-06feaf6dde2b"
        }
    }
)

for message in result["messages"]:
    message.pretty_print() 

================================ Human Message =================================

Hello my name is Hakyung
================================== Ai Message ==================================

Hello, Hakyung! How can I assist you today?
================================ Human Message =================================

I ive in Korea now. And the city I live in gyeonggi-do
================================== Ai Message ==================================

That's great! Gyeonggi-do is a beautiful province with a lot to offer, from its rich history to modern attractions. What do you enjoy most about living there?
